In [2]:
import requests, feedparser, json,time
import polars as pl
from config_info import APIS
from pprint import pprint
from lxml import etree

# Basic Fetch

In [3]:
def fetch_raw(url, headers=None):
    headers = headers or {"User-Agent":  "IntelliCorpus/1.0 (contact: paull@scholar-cergy.com)"}
    r = requests.get(url, headers=headers, timeout=15)
    r.raise_for_status()
    content_type = r.headers.get("Content-Type","")
    if 'xml' in content_type:
        return r.text
    elif 'json' in content_type:
        return r.json()
    else:
        return r.text

# Arxiv 

In [4]:
def parser_arxiv(raw_result : str):
    feed = feedparser.parse(raw_result)
    results = []
    for e in feed.entries:
        results.append({
            "id": e.get("id"),
            "title": e.get("title"),
            "summary": e.get("summary"),
            "published": e.get("published"),
            "updated": e.get("updated"),
            "authors": [a.name for a in e.get("authors", [])],
            "pdf_url": next((link.href for link in e.links if link.type == "application/pdf"), None),
            "source": "arxiv"
        })
    return results

# Hal

In [ ]:
def get_pdf_hal(link : str):
    

In [5]:
def parser_hal(feed : str):
    if isinstance(feed, str):
        feed = json.loads(feed)
    docs = feed.get("response", {}).get("docs", [])
    results = []
    for doc in docs:
        results.append({
            "id": doc.get("docid"),
            "title" : doc.get("title_s"),
            "summary" : doc.get("abstract_s"),
            "published" : doc.get("publicationDate_s"),
            "authors" : [a for a in doc.get("authFullName_s",[])],
            "uri": doc.get("uri_s"),
            "pdf_url" : doc.get("files_s"),
            "source": "hal"
        })
    return results

# Pubmed        

In [6]:

from typing import List, Dict
EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
DB = "pubmed"


def pubmed_search(
    query: str,
    retmax: int = 100,
    retstart: int = 0,
    email: str | None = None,
    api_key: str | None = None) -> Dict:
    """
    Step 1: Search PubMed and store results on NCBI server (usehistory)
    """
    url = f"{EUTILS_BASE}/esearch.fcgi"

    params = {
        "db": DB,
        "term": query,
        "retmax": retmax,
        "retstart": retstart,
        "usehistory": "y",
        "retmode": "json",
    }

    if email:
        params["email"] = email
    if api_key:
        params["api_key"] = api_key

    response = requests.get(url, params=params, timeout=15)
    response.raise_for_status()
    return response.json()["esearchresult"]


def pubmed_fetch(
    webenv: str,
    query_key: str,
    batch_size: int = 100,
    email: str | None = None,
    api_key: str | None = None
) -> str:
    """
    Step 2: Fetch PubMed records using WebEnv + QueryKey (XML)
    """
    url = f"{EUTILS_BASE}/efetch.fcgi"

    params = {
        "db": DB,
        "query_key": query_key,
        "WebEnv": webenv,
        "retmax": batch_size,
        "retmode": "xml",
    }

    if email:
        params["email"] = email
    if api_key:
        params["api_key"] = api_key

    response = requests.get(url, params=params, timeout=20)
    response.raise_for_status()
    return response.text


def fetch_pubmed(query: str, max_results: int = 200, batch_size: int = 100, delay: float = 0.34, email: str | None = None, api_key: str | None = None) -> List[str]:
    """
    High-level generator:
    - search
    - iterate over result pages
    - fetch XML batches
    """
    search_result = pubmed_search(
        query=query,
        retmax=max_results,
        email=email,
        api_key=api_key
    )
    count = int(search_result["count"])
    webenv = search_result["webenv"]
    query_key = search_result["querykey"]

    results_xml = []

    for start in range(0, min(count, max_results), batch_size):
        xml = pubmed_fetch(
            webenv=webenv,
            query_key=query_key,
            batch_size=batch_size,
            email=email,
            api_key=api_key,
        )
        results_xml.append(xml)
        time.sleep(delay)  # respect NCBI rate-limit
        # pprint(xml)
    return results_xml


# Main 

In [13]:
# raw_result = fetch_raw(APIS["arXiv"]["api_url"].format(query="AI agent",quantity='1'))
# result_arxiv =  parser_arxiv(raw_result)
# print(result_arxiv) # Arxiv Ok 

# Test HAL
url_hal = fetch_raw(APIS["HAL"]["api_url"].format(query="AI agent",quantity='2'))
result_hal = parser_hal(url_hal)
# pprint(result_hal) Good 

# Test PubMed
xml_batches = fetch_pubmed(
    query="AI agent",
    max_results=50,
    batch_size=20,
    email="proliquer@scholar-perigueuxu.com"
)
pprint(xml_batches)




['<?xml version="1.0" ?>\n'
 '<!DOCTYPE PubmedArticleSet PUBLIC "-//NLM//DTD PubMedArticle, 1st January '
 '2025//EN" "https://dtd.nlm.nih.gov/ncbi/pubmed/out/pubmed_250101.dtd">\n'
 '<PubmedArticleSet>\n'
 '<PubmedArticle><MedlineCitation Status="MEDLINE" Owner="NLM" '
 'IndexingMethod="Automated"><PMID '
 'Version="1">41582762</PMID><DateCompleted><Year>2026</Year><Month>01</Month><Day>26</Day></DateCompleted><DateRevised><Year>2026</Year><Month>01</Month><Day>26</Day></DateRevised><Article '
 'PubModel="Print"><Journal><ISSN '
 'IssnType="Electronic">1612-1880</ISSN><JournalIssue '
 'CitedMedium="Internet"><Volume>23</Volume><Issue>1</Issue><PubDate><Year>2026</Year><Month>Jan</Month></PubDate></JournalIssue><Title>Chemistry '
 '&amp; biodiversity</Title><ISOAbbreviation>Chem '
 'Biodivers</ISOAbbreviation></Journal><ArticleTitle>Synthesis, In Vitro '
 'Antimicrobial, and Antioxidant Activities of Novel Thiazole '
 'Derivatives.</ArticleTitle><Pagination><StartPage>e03308</StartPage